In [2]:
import os
import re
from datetime import datetime

def delete_copy_files(directory, dry_run: bool = True, print_flag: bool = False):
    # Regex to catch naming like file_copy1.xlsx or file_copy2.xls
    files_removed_count = 0
    files_to_remove_count = 0
    files_close_enough = 0
    files_nearly_identical = 0
    files_very_different = 0
    files_size_to_reduce = 0
    copy_pattern = re.compile(r"^(?P<name>.+)_copy\d+\.(?P<ext>xlsx|xls)$", re.IGNORECASE)

    # Indexing files by base name
    base_files = {}
    for filename in os.listdir(directory):
        if not filename.lower().endswith(('.xlsx', '.xls')):
            continue

        filepath = os.path.join(directory, filename)
        match = copy_pattern.match(filename)

        if match:
            base_name = f"{match.group('name')}.{match.group('ext')}"
        else:
            base_name = filename

        base_files.setdefault(base_name, []).append(filepath)

    for base_name, paths in base_files.items():
        if len(paths) < 2:
            continue  # no duplicates

        # Sort versions by modification date (most recent last)
        paths.sort(key=lambda p: os.path.getmtime(p))

        keep_path = paths[-1]  # start from newest as default
        keep_size = os.path.getsize(keep_path)

        for p in paths[:-1]:
            size_diff = abs(keep_size - os.path.getsize(p))

            if size_diff < 10 * 1024:  # less than 10KB
                if print_flag:
                    print(f"Deleting {p} (nearly identical to {keep_path})")
                files_to_remove_count += 1
                files_nearly_identical += 1
                files_size_to_reduce += os.path.getsize(p)
                if not dry_run:
                    try:
                        os.remove(p)
                        files_removed_count += 1
                    except OSError as e:
                        print(f"  [ERROR] Could not delete {p}: {e}")
            elif size_diff > 1 * 1024 * 1024:  # greater than 1MB
                files_very_different += 1
                if print_flag:
                    print(f"Keeping both {p} and {keep_path} (significant size difference)")
                continue
            else:  # between 10KB and 1MB
                # keep the larger one
                files_close_enough+=1
                if os.path.getsize(p) > keep_size:
                    if print_flag:
                        print(f"Deleting smaller file {keep_path}, keeping {p}")
                    files_to_remove_count += 1
                    files_size_to_reduce += os.path.getsize(p)
                    if not dry_run:
                        try:
                            os.remove(keep_path)
                            files_removed_count += 1
                        except OSError as e:
                            print(f"  [ERROR] Could not delete {keep_path}: {e}")
                    keep_path = p
                    keep_size = os.path.getsize(p)
                else:
                    if print_flag:
                        print(f"Deleting smaller file {p}, keeping {keep_path}")
                    files_to_remove_count += 1
                    files_size_to_reduce += os.path.getsize(p)
                    if not dry_run:
                        try:
                            os.remove(p)
                            files_removed_count += 1
                        except OSError as e:
                            print(f"  [ERROR] Could not delete {p}: {e}")
    return {"files_removed": files_removed_count,"files_to_remove": files_to_remove_count, "files_identical":files_nearly_identical,"files_different": files_very_different, "files_almost_similar": files_close_enough, "Clear_size(MB)": round(files_size_to_reduce*1e-6)}

# if __name__ == "__main__":
#     folder_path = input("Enter the directory path to clean: ").strip()
#     if os.path.isdir(folder_path):
#         count = delete_copy_files(folder_path, dry_run=True, print_flag=True)
#         print(f"{count} files deleted")
#     else:
#         print("Invalid directory path.")


In [12]:
import glob
import pandas as pd

out = []
DRY_RUN = False
samples = ["B2_sample"]
data_folders = ["02_Extracted_Raw_Files"]
basepath = r"\\?\UNC\volkswagengroup.sharepoint.com@SSL\DavWWWRoot\sites\C48Testdata2\Shared Documents\C48T1\02_CPA_DataBase\x_DataHandling"

if DRY_RUN:
    print("Dry Run active, no files will be deleted:")
else:
    flag = input("!! Warning, do you want to delete these files? (y/n)")
    if flag == "y":
        DRY_RUN = False
    else:
        DRY_RUN = True
        print("Dry Run activated, no files will be deleted:")

for sample in samples:
    for data_folder in data_folders:
        paths = glob.glob(basepath+f'\\{sample}\\{data_folder}\\*')
        for path in paths:
            count_dict = delete_copy_files(path, dry_run = DRY_RUN, print_flag = False)
            out.append(["\\".join(path.split("\\")[-6:-3]),path.split("\\")[-2],path.split("\\")[-3],path.split("\\")[-1],count_dict["files_removed"], count_dict["files_to_remove"], count_dict["files_identical"], count_dict["files_different"], count_dict["files_almost_similar"], count_dict["Clear_size(MB)"]])
out = pd.DataFrame(out,columns=["BasePath", "Category", "Sample", "CellID", "Removed","to_remove","Identical","Almost_same","Different","Clear_size(MB)"])

print(out[["Sample","Clear_size(MB)"]].groupby("Sample").sum())

           Clear_size(MB)
Sample                   
B2_sample            3906


In [1]:
import os
from typing import Iterable, List, Tuple

def delete_small_files(
    root_folder: str,
    threshold_bytes: int = 1,
    extensions: Iterable[str] = ("csv", "xlsx"),
    dry_run: bool = True,
) -> Tuple[List[str], List[Tuple[str, str]]]:
    """
    Delete files in the folder and subfolders if:
      - Their file size is <= threshold_bytes
      - Their extension is in 'extensions'

    Supports a dry-run mode to preview actions.

    Args:
        root_folder (str): Path to the folder to scan.
        threshold_bytes (int): Max allowed size (in bytes) to delete (inclusive).
        extensions (Iterable[str]): File extensions to include (e.g., ['csv', 'xlsx']).
        dry_run (bool): If True, do not delete—only report. If False, delete.

    Returns:
        Tuple[List[str], List[Tuple[str, str]]]:
            - deleted_or_would_delete: List of file paths deleted (or that would be deleted in dry-run).
            - errors: List of (file_path, error_message) for any files that couldn't be processed.

    Notes:
        - Extension match is case-insensitive and ignores leading dots.
        - Uses os.walk to traverse subdirectories.
        - Only files with size <= threshold are targeted.
    """
    if not os.path.isdir(root_folder):
        raise ValueError(f"Provided root_folder is not a directory or doesn't exist: {root_folder}")

    # Normalize extensions (remove dot, lowercase)
    normalized_exts = {str(ext).lower().lstrip(".") for ext in extensions} if extensions else set()

    deleted_or_would_delete: List[str] = []
    errors: List[Tuple[str, str]] = []

    for dirpath, _, filenames in os.walk(root_folder):
        for filename in filenames:
            file_path = os.path.join(dirpath, filename)

            # Safely determine extension
            _, ext = os.path.splitext(filename)
            ext = ext.lower().lstrip(".")  # '.CSV' -> 'csv'

            if normalized_exts and ext not in normalized_exts:
                continue

            # Get size and decide
            try:
                size = os.path.getsize(file_path)
            except OSError as e:
                errors.append((file_path, f"stat failed: {e}"))
                continue

            if size <= threshold_bytes:
                deleted_or_would_delete.append(file_path)
                if not dry_run:
                    try:
                        os.remove(file_path)
                    except OSError as e:
                        errors.append((file_path, f"delete failed: {e}"))

    return deleted_or_would_delete, errors

In [7]:
deleted, errors = delete_small_files(
    root_folder=r"C:\Users\WYBT00P\OneDrive - Volkswagen AG\C48 Test data #2 - 02_CPA_DataBase\x_DataHandling\B2_sample\02_Extracted_Raw_Files",
    threshold_bytes=1,
    extensions=("csv", "xlsx"),
    dry_run=False,  # Actually delete
)

print(f"Deleted {len(deleted)} file(s).")
if errors:
    print("\nErrors encountered:")
    for p, msg in errors:
        print(f"{p} -> {msg}")

Deleted 0 file(s).

Errors encountered:
C:\Users\WYBT00P\OneDrive - Volkswagen AG\C48 Test data #2 - 02_CPA_DataBase\x_DataHandling\B2_sample\02_Extracted_Raw_Files\DQ202512003253\LAB-VW-CVD-1142_2_DQ202512003253-EPV20252220_-EPV20252220--215.85AH C48T-(2%_100%)SOC区间循环  2.5-3.8 14D_20251222171839_part1.xlsx -> stat failed: [WinError 3] The system cannot find the path specified: 'C:\\Users\\WYBT00P\\OneDrive - Volkswagen AG\\C48 Test data #2 - 02_CPA_DataBase\\x_DataHandling\\B2_sample\\02_Extracted_Raw_Files\\DQ202512003253\\LAB-VW-CVD-1142_2_DQ202512003253-EPV20252220_-EPV20252220--215.85AH C48T-(2%_100%)SOC区间循环  2.5-3.8 14D_20251222171839_part1.xlsx'
C:\Users\WYBT00P\OneDrive - Volkswagen AG\C48 Test data #2 - 02_CPA_DataBase\x_DataHandling\B2_sample\02_Extracted_Raw_Files\DQ202512003253\LAB-VW-CVD-1142_2_DQ202512003253-EPV20252220_-EPV20252220--215.85AH C48T-(2%_100%)SOC区间循环  2.5-3.8 14D_20251222171839_part2.xlsx -> stat failed: [WinError 3] The system cannot find the path specified

In [45]:
import plotly.express as px
px.

'C48 Test data - C48T1\\02_CPA_DataBase\\x_DataHandling'